[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/graph_theory/07_spectral_clustering_and_gnn_applications/first_principles.ipynb)

# Topic 07: Spectral Clustering and GNN Applications

## 1. First-Principles Intuition & Motivation

Two blobs of points in the plane are easy to separate: draw a line. Two *concentric rings* are not — no line works, and $k$-means, which carves space into convex cells around centroids, fails completely. Yet the rings are obviously two clusters, because points on the same ring are connected by a chain of near neighbours while points on different rings are not.

That observation is the whole idea. Replace "distance in $\mathbb{R}^d$" by "connectivity in a similarity graph", and clustering becomes **graph partitioning**: cut the graph into pieces, cutting as few edges as possible while keeping the pieces balanced. Topic 06 supplied the tool for exactly this question — the Laplacian, whose quadratic form $x^{\top}Lx = \sum_{\{u,v\} \in E} w_{uv}(x_u - x_v)^2$ is small precisely for signals that do not change across edges.

The obstacle is that the balanced-cut problem is NP-hard. The resolution is one of the most productive moves in applied mathematics: **relax, solve, round**. Write the cut objective as a quadratic form over *discrete* indicator vectors; drop the discreteness constraint; the resulting continuous problem is an eigenvalue problem solvable exactly; then round the continuous solution back to a partition. The relaxed optimum is a Laplacian eigenvector, the rounding is $k$-means, and the resulting pipeline is spectral clustering.

The same relaxation-free machinery, used forward instead of backward, gives graph neural networks. If low Laplacian eigenvalues mean "smooth on the graph", then multiplying by a polynomial in $L$ is a **filter** that keeps or removes graph frequencies. A graph convolutional layer is the cheapest possible such filter — degree one — followed by a linear map and a nonlinearity. Clustering reads structure *out of* the operator; GNNs push features *through* it.

### The picture in one paragraph

Suppose the graph is two dense clusters joined by a few edges. The indicator vector of one cluster is *almost* in the kernel of $L$: it is constant on each cluster, so it pays energy only on the few crossing edges. As the connecting edges vanish, that vector becomes exactly a kernel vector and $\lambda_2 \to 0$ (Topic 06's kernel theorem). For a nearly-disconnected graph, then, the eigenvector of $\lambda_2$ must be close to a cluster indicator — approximately constant on each cluster, changing sign between them. Thresholding it recovers the clusters. Everything below is that intuition made precise, generalized from $2$ to $k$ clusters, and equipped with the correct balancing term.

### Why a *balancing* term is unavoidable

Plain minimum cut is solvable in polynomial time (Topic 05), but it is useless for clustering: the cheapest cut usually shaves off a single low-degree vertex. Two standard repairs divide the cut by a size measure:

$$
\mathrm{RatioCut}(A_1,\dots,A_k) = \sum_{i=1}^{k}\frac{\mathrm{cut}(A_i, \bar{A_i})}{\vert A_i \vert}, \qquad \mathrm{NCut}(A_1,\dots,A_k) = \sum_{i=1}^{k}\frac{\mathrm{cut}(A_i, \bar{A_i})}{\mathrm{vol}(A_i)}
$$

RatioCut balances the *number of vertices*; NCut balances the *volume* (sum of degrees). Both are NP-hard — and both become tractable after relaxation, RatioCut leading to the unnormalized Laplacian $L$ and NCut to the normalized ones $L_{\mathrm{rw}}, L_{\mathrm{sym}}$. Which balancing you want is a modelling decision, not a technical one: NCut is the right choice whenever degrees vary widely, which in practice is almost always.

## 2. Rigorous Mathematical Definitions & Theorem Statements

Let $G = (V, E)$ be a weighted undirected graph with $n$ vertices, weights $w_{uv} \ge 0$, degree matrix $D$, Laplacian $L = D - A$, and normalized Laplacians $L_{\mathrm{sym}} = D^{-1/2}LD^{-1/2}$, $L_{\mathrm{rw}} = D^{-1}L = I - P$ with $P = D^{-1}A$.

**Definition (Similarity graph).** Given data $x_1, \dots, x_n$ and a similarity $s(x_i, x_j) \ge 0$, three standard constructions:

- **$\varepsilon$-neighbourhood graph**: connect $i \sim j$ iff $\Vert x_i - x_j \Vert \le \varepsilon$ (usually unweighted).
- **$k$-nearest-neighbour graph**: connect $i \to j$ if $j$ is among $i$'s $k$ nearest neighbours, then symmetrize (union: the *kNN graph*; intersection: the *mutual kNN graph*).
- **Fully connected graph** with the Gaussian (RBF) kernel $w_{ij} = \exp\big(-\Vert x_i - x_j\Vert^2 / (2\sigma^2)\big)$.

**Definition (Cut, volume, objectives).** For a partition $A_1, \dots, A_k$ of $V$: $\mathrm{cut}(A, B) = \sum_{u \in A, v \in B} w_{uv}$, $\ \mathrm{vol}(A) = \sum_{v \in A}d_v$, and RatioCut / NCut as displayed above. For $k = 2$, minimizing NCut is equivalent (up to a factor $2$) to minimizing the conductance $h$ of Topic 06.

**Definition (Spectral embedding).** Let $u_1, \dots, u_k$ be eigenvectors of the chosen Laplacian for its $k$ smallest eigenvalues and $U = [u_1 \cdots u_k] \in \mathbb{R}^{n \times k}$. The **spectral embedding** maps vertex $v$ to the $v$-th row $y_v = U_{v,\cdot} \in \mathbb{R}^{k}$.

**Definition (The three canonical algorithms).** All take a similarity graph and a target $k$, and end with $k$-means on the embedded points.

| Algorithm | Matrix diagonalized | Embedding | Source |
|---|---|---|---|
| Unnormalized spectral clustering | $L$ | rows of $U$ | von Luxburg §4 |
| Normalized, random-walk (**Shi–Malik**) | generalized problem $Lu = \lambda D u$, i.e. $L_{\mathrm{rw}}$ | rows of $U$ | Shi & Malik (2000) |
| Normalized, symmetric (**Ng–Jordan–Weiss**) | $L_{\mathrm{sym}}$ | rows of $U$ rescaled to unit norm | Ng, Jordan & Weiss (2002) |

**Definition (Graph Fourier transform and spectral filter).** With $L = U\Lambda U^{\top}$, the transform is $\hat{x} = U^{\top}x$, and a **spectral filter** with response $g_\theta$ acts as

$$
g_\theta \star x = U\, g_\theta(\Lambda)\, U^{\top} x = g_\theta(L)\, x
$$

**Definition (Chebyshev polynomials).** $T_0(y) = 1$, $T_1(y) = y$, $T_{k+1}(y) = 2yT_k(y) - T_{k-1}(y)$, orthogonal on $[-1,1]$; the rescaled Laplacian is $\tilde{L} = \frac{2}{\lambda_{\max}}L_{\mathrm{sym}} - I$, whose spectrum lies in $[-1,1]$.

**Definition (GCN layer).** With $\tilde{A} = A + I$, $\tilde{D} = \mathrm{diag}(\tilde{A}\mathbf{1})$, and $\hat{S} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$:

$$
H^{(l+1)} = \sigma\big(\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2} H^{(l)} W^{(l)}\big) = \sigma\big(\hat{S}H^{(l)}W^{(l)}\big)
$$

with $H^{(0)} = X$ the input features and $W^{(l)}$ learned weights.

**Definition (Message passing).** A general layer is

$$
h_v^{(l+1)} = \phi\Big(h_v^{(l)}, \ \bigoplus_{u \in N(v)} \psi\big(h_v^{(l)}, h_u^{(l)}, e_{uv}\big)\Big)
$$

with a permutation-invariant aggregator $\bigoplus$ (sum, mean, max). The GCN layer is the special case $\psi(h_v, h_u) = \tilde{c}_{uv} h_u$ with $\tilde{c}_{uv} = (\tilde{d}_u \tilde{d}_v)^{-1/2}$, $\bigoplus = \sum$, and $\phi = \sigma(\cdot\, W)$.

### Theorem statements

**T1 (RatioCut relaxation, $k = 2$).** For a bipartition $(A, \bar{A})$ define $f \in \mathbb{R}^n$ by $f_v = \sqrt{\vert \bar{A}\vert / \vert A \vert}$ for $v \in A$ and $f_v = -\sqrt{\vert A \vert/\vert \bar{A}\vert}$ otherwise. Then $f \perp \mathbf{1}$, $\Vert f \Vert^2 = n$, and

$$
f^{\top} L f = n \cdot \mathrm{RatioCut}(A, \bar{A})
$$

Relaxing $f$ to range over all of $\mathbf{1}^{\perp}$ makes the minimum $n\lambda_2$, attained at the Fiedler vector.

**T2 (RatioCut relaxation, general $k$).** With $H \in \mathbb{R}^{n\times k}$, $H_{vi} = \vert A_i\vert^{-1/2}\mathbf{1}[v \in A_i]$, one has $H^{\top}H = I_k$ and $\mathrm{RatioCut} = \operatorname{tr}(H^{\top}LH)$. Relaxing to arbitrary $H$ with $H^{\top}H = I_k$ yields $\min = \sum_{i=1}^{k}\lambda_i(L)$, attained by the bottom $k$ eigenvectors.

**T3 (Ky Fan / Rayleigh–Ritz).** For symmetric $M$ with eigenvalues $\lambda_1 \le \cdots \le \lambda_n$,

$$
\min_{H^{\top}H = I_k} \operatorname{tr}(H^{\top}MH) = \sum_{i=1}^{k}\lambda_i
$$

**T4 (NCut relaxation).** With $H_{vi} = \mathrm{vol}(A_i)^{-1/2}\mathbf{1}[v \in A_i]$ one has $H^{\top}DH = I_k$ and $\mathrm{NCut} = \operatorname{tr}(H^{\top}LH)$; the relaxation $\min\{\operatorname{tr}(H^{\top}LH) : H^{\top}DH = I\}$ is solved by the bottom $k$ eigenvectors of the generalized problem $Lu = \lambda Du$, equivalently of $L_{\mathrm{rw}}$, equivalently by $H = D^{-1/2}T$ with $T$ the bottom eigenvectors of $L_{\mathrm{sym}}$.

**T5 (Random-walk interpretation).** For the stationary random walk with $\pi_v = d_v/\mathrm{vol}(V)$,

$$
\mathrm{NCut}(A, \bar{A}) = \Pr\big[X_1 \in \bar{A} \mid X_0 \in A\big] + \Pr\big[X_1 \in A \mid X_0 \in \bar{A}\big]
$$

**T6 (GCN as a first-order spectral filter).** Truncating the Chebyshev expansion of $g_\theta$ at $K = 1$, setting $\lambda_{\max} \approx 2$ and tying the two coefficients gives $g_\theta \star x \approx \theta\big(I + D^{-1/2}AD^{-1/2}\big)x$; the renormalization $I + D^{-1/2}AD^{-1/2} \rightsquigarrow \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ produces the GCN propagation rule.

**T7 (Over-smoothing).** For a connected graph, $\hat{S} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ has eigenvalues $1 = \mu_1 \gt \mu_2 \ge \cdots \ge \mu_n \gt -1$, and

$$
\hat{S}^{\,l} \longrightarrow \frac{\tilde{D}^{1/2}\mathbf{1}\,\mathbf{1}^{\top}\tilde{D}^{1/2}}{\mathbf{1}^{\top}\tilde{D}\mathbf{1}} \quad \text{as } l \to \infty, \qquad \text{at rate } \vert \mu_2 \vert^{\,l}
$$

so a deep stack of purely linear propagations collapses all node representations onto one direction.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1: RatioCut for two clusters relaxes to the Fiedler vector (T1)

**Setup.** Fix a bipartition $(A, \bar{A})$ with $A \neq \emptyset, V$, and define

$$
f_v = \begin{cases} \sqrt{\vert \bar{A}\vert / \vert A \vert} & v \in A \\ -\sqrt{\vert A \vert / \vert \bar{A} \vert} & v \in \bar{A} \end{cases}
$$

**Step 1 — $f$ is centred.** $\sum_v f_v = \vert A\vert\sqrt{\vert \bar A\vert/\vert A\vert} - \vert \bar A\vert\sqrt{\vert A\vert/\vert \bar A\vert} = \sqrt{\vert A\vert \, \vert \bar A\vert} - \sqrt{\vert A\vert \, \vert \bar A\vert} = 0$, so $f \perp \mathbf{1}$.

**Step 2 — $f$ has fixed norm.** $\Vert f\Vert^2 = \vert A\vert\frac{\vert \bar A\vert}{\vert A\vert} + \vert \bar A\vert\frac{\vert A\vert}{\vert \bar A\vert} = \vert \bar A\vert + \vert A\vert = n$.

**Step 3 — the energy is the objective.** By the energy identity, only cut edges contribute, each with difference $\sqrt{\vert \bar A\vert/\vert A\vert} + \sqrt{\vert A\vert/\vert \bar A\vert}$:

$$
f^{\top}Lf = \mathrm{cut}(A,\bar A)\left(\sqrt{\frac{\vert \bar A\vert}{\vert A\vert}} + \sqrt{\frac{\vert A\vert}{\vert \bar A\vert}}\right)^{2} = \mathrm{cut}(A,\bar A)\left(\frac{\vert \bar A\vert}{\vert A\vert} + 2 + \frac{\vert A\vert}{\vert \bar A\vert}\right)
$$

Group the terms as $\big(\frac{\vert \bar A\vert}{\vert A\vert} + 1\big) + \big(\frac{\vert A\vert}{\vert \bar A\vert} + 1\big) = \frac{n}{\vert A\vert} + \frac{n}{\vert \bar A\vert}$, so

$$
f^{\top}Lf = n\,\mathrm{cut}(A,\bar A)\left(\frac{1}{\vert A\vert} + \frac{1}{\vert \bar A\vert}\right) = n\,\mathrm{RatioCut}(A, \bar{A})
$$

**Step 4 — relax and solve.** Minimizing RatioCut is therefore *exactly*

$$
\min_{A \subset V} f^{\top}Lf \quad \text{s.t.} \quad f \perp \mathbf{1}, \ \Vert f\Vert^2 = n, \ f \text{ of the two-valued form above}
$$

Dropping the last (combinatorial) constraint gives a continuous problem whose solution, by Courant–Fischer (Topic 06), is $\lambda_2$ with minimizer the Fiedler vector:

$$
\boxed{\min_{f \perp \mathbf{1}, \ \Vert f \Vert^2 = n} f^{\top}Lf = n\,\lambda_2 \quad \Longrightarrow \quad \mathrm{RatioCut}^{\star} \ge \lambda_2}
$$

**Step 5 — round.** The relaxed minimizer is real-valued, so a partition is recovered by thresholding: sign, median, or the best of the $n-1$ sweep cuts. The Cheeger inequality (Topic 06) bounds the quality of the sweep-cut rounding for the normalized objective; for RatioCut no comparable general guarantee exists, and Guattery & Miller (1998) exhibit graphs where the relaxation is off by a factor $\Omega(n)$. $\blacksquare$

### Proof 2: The trace form for $k$ clusters, and Ky Fan's theorem (T2, T3)

**Encoding a partition.** Define $H \in \mathbb{R}^{n \times k}$ by

$$
H_{vi} = \begin{cases} 1/\sqrt{\vert A_i\vert} & v \in A_i \\ 0 & \text{otherwise} \end{cases}
$$

The columns have disjoint supports and unit norm, so $H^{\top}H = I_k$.

**Column energies are the objective's terms.** For column $h_i$, the energy identity gives contributions only from edges leaving $A_i$, each of size $(1/\sqrt{\vert A_i\vert} - 0)^2$:

$$
h_i^{\top}Lh_i = \sum_{\{u,v\} \in E} w_{uv}(H_{ui} - H_{vi})^2 = \frac{\mathrm{cut}(A_i, \bar{A_i})}{\vert A_i\vert}
$$

Since $h_i^{\top}Lh_i = (H^{\top}LH)_{ii}$, summing over $i$ yields

$$
\operatorname{tr}(H^{\top}LH) = \sum_{i=1}^{k}\frac{\mathrm{cut}(A_i,\bar{A_i})}{\vert A_i\vert} = \mathrm{RatioCut}(A_1,\dots,A_k)
$$

**Ky Fan's theorem (full proof).** Let $M$ be symmetric with orthonormal eigenpairs $(\lambda_j, u_j)$, $\lambda_1 \le \cdots \le \lambda_n$, and let $H^{\top}H = I_k$. Write $M = \sum_j \lambda_j u_ju_j^{\top}$; then

$$
\operatorname{tr}(H^{\top}MH) = \sum_{j=1}^{n}\lambda_j \operatorname{tr}(H^{\top}u_ju_j^{\top}H) = \sum_{j=1}^{n}\lambda_j \Vert H^{\top}u_j\Vert^2 = \sum_{j=1}^{n}\lambda_j c_j
$$

with $c_j = \Vert H^{\top}u_j\Vert^2 = u_j^{\top}HH^{\top}u_j$. Because $HH^{\top}$ is an orthogonal projector of rank $k$:

- $0 \le c_j \le 1$ for every $j$;
- $\sum_j c_j = \operatorname{tr}(HH^{\top}) = \operatorname{tr}(H^{\top}H) = k$.

Minimizing the linear functional $\sum_j \lambda_j c_j$ over the polytope $\{c \in [0,1]^n : \sum_j c_j = k\}$ puts all available mass on the $k$ smallest $\lambda_j$, giving

$$
\operatorname{tr}(H^{\top}MH) \ge \sum_{i=1}^{k}\lambda_i
$$

with equality when $H = [u_1 \cdots u_k]Q$ for any orthogonal $Q$. $\blacksquare$

$$
\boxed{\min_{H^{\top}H = I_k}\operatorname{tr}(H^{\top}LH) = \sum_{i=1}^{k}\lambda_i(L), \ \text{ attained by the bottom } k \text{ eigenvectors}}
$$

**Consequence.** The relaxed RatioCut optimum is $\sum_{i \le k}\lambda_i$, and the relaxed solution is *any* orthonormal basis of the bottom-$k$ eigenspace — which is why the embedding is only defined up to a rotation $Q$, and why the rounding step must be rotation-invariant. $k$-means is.

### Proof 3: NCut relaxes to the normalized Laplacians (T4)

**Encoding.** Now weight by volume:

$$
H_{vi} = \begin{cases} 1/\sqrt{\mathrm{vol}(A_i)} & v \in A_i \\ 0 & \text{otherwise} \end{cases}
$$

Then $(H^{\top}DH)_{ii} = \sum_{v \in A_i} d_v/\mathrm{vol}(A_i) = 1$ and off-diagonal entries vanish by disjoint support, so $H^{\top}DH = I_k$. Exactly as in Proof 2,

$$
(H^{\top}LH)_{ii} = \frac{\mathrm{cut}(A_i,\bar{A_i})}{\mathrm{vol}(A_i)} \quad \Longrightarrow \quad \operatorname{tr}(H^{\top}LH) = \mathrm{NCut}(A_1,\dots,A_k)
$$

**Relaxation.** Minimize $\operatorname{tr}(H^{\top}LH)$ subject to $H^{\top}DH = I_k$ over all real $H$. Substitute $T = D^{1/2}H$, so that $T^{\top}T = I_k$ and

$$
\operatorname{tr}(H^{\top}LH) = \operatorname{tr}\big(T^{\top}D^{-1/2}LD^{-1/2}T\big) = \operatorname{tr}(T^{\top}L_{\mathrm{sym}}T)
$$

By Ky Fan this is minimized by taking $T$ to be the bottom $k$ eigenvectors of $L_{\mathrm{sym}}$, with value $\sum_{i \le k}\mu_i$. Undoing the substitution, $H = D^{-1/2}T$ consists of the bottom $k$ eigenvectors of the generalized problem

$$
Lu = \lambda D u \iff D^{-1}Lu = \lambda u \iff L_{\mathrm{rw}}u = \lambda u
$$

$\blacksquare$

$$
\boxed{\text{NCut relaxation} \ \Longleftrightarrow \ \text{bottom eigenvectors of } L_{\mathrm{rw}}, \quad u = D^{-1/2}t \text{ with } L_{\mathrm{sym}}t = \mu t}
$$

**Reading the three algorithms off this proof.**

- **Unnormalized** clustering embeds with eigenvectors of $L$ — the RatioCut relaxation.
- **Shi–Malik** embeds with $u = D^{-1/2}t$, the $L_{\mathrm{rw}}$ eigenvectors — the *exact* NCut relaxation.
- **Ng–Jordan–Weiss** embeds with $t$ itself (eigenvectors of $L_{\mathrm{sym}}$) and then renormalizes each row to unit length — an approximate undoing of the $D^{-1/2}$ that Shi–Malik applies exactly.

The three differ only in how the degree factor $D^{-1/2}$ is handled, which is precisely why they agree on regular graphs and diverge on heavy-tailed ones.

### Proof 4: NCut is an escape probability (T5)

**Theorem.** Let $(X_t)$ be the random walk with $P = D^{-1}A$, started from the stationary distribution $\pi_v = d_v/\mathrm{vol}(V)$. Then for any $A \subseteq V$,

$$
\Pr[X_1 \in \bar{A} \mid X_0 \in A] = \frac{\mathrm{cut}(A, \bar{A})}{\mathrm{vol}(A)}
$$

and consequently $\mathrm{NCut}(A,\bar A)$ is the total probability of the walk crossing the boundary in one step, conditioned on each side.

**Proof.** Compute the joint probability directly:

$$
\Pr[X_0 \in A, X_1 \in \bar{A}] = \sum_{u \in A}\sum_{v \in \bar{A}}\pi_u P_{uv} = \sum_{u \in A}\sum_{v \in \bar{A}}\frac{d_u}{\mathrm{vol}(V)}\cdot\frac{w_{uv}}{d_u} = \frac{\mathrm{cut}(A,\bar A)}{\mathrm{vol}(V)}
$$

The degree cancels — that cancellation is the whole content of the theorem. Since $\Pr[X_0 \in A] = \mathrm{vol}(A)/\mathrm{vol}(V)$, dividing gives the conditional probability $\mathrm{cut}(A,\bar A)/\mathrm{vol}(A)$. Adding the symmetric term for $\bar{A}$:

$$
\boxed{\mathrm{NCut}(A,\bar A) = \Pr[X_1 \in \bar A \mid X_0 \in A] + \Pr[X_1 \in A \mid X_0 \in \bar A]}
$$

$\blacksquare$

**Interpretation.** A good NCut partition is one the random walk rarely leaves: clusters are **metastable** sets. This connects spectral clustering to Markov chain theory — small eigenvalues of $L_{\mathrm{rw}} = I - P$ are eigenvalues of $P$ close to $1$, i.e. slowly decaying modes, and the number of such modes is the natural number of clusters. It also explains the eigengap heuristic: choose $k$ where $\lambda_{k+1} - \lambda_k$ is large, because that is where "slow, cluster-like" modes end and "fast, within-cluster" modes begin.

**Commute-time view.** The same walk gives the commute-time embedding $y_v = \Lambda^{-1/2}U^{\top}e_v$, under which squared Euclidean distance equals commute time (up to $2m$); spectral clustering is approximately $k$-means in commute-time geometry, where "close" means "the walk moves between them quickly".

### Proof 5: From spectral filters to the GCN propagation rule (T6)

**Step 1 — spectral convolution.** Convolution on a graph is defined in the Fourier domain: for a filter with response $g_\theta$,

$$
g_\theta \star x = U g_\theta(\Lambda)U^{\top}x = g_\theta(L)x
$$

Learning $g_\theta(\Lambda) = \mathrm{diag}(\theta_1,\dots,\theta_n)$ freely costs $n$ parameters, requires the full eigendecomposition ($O(n^3)$), and produces filters with **global** support — a vertex's output depends on the entire graph. All three are unacceptable.

**Step 2 — polynomial filters are localized.** Restrict to polynomials, $g_\theta(L) = \sum_{k=0}^{K}\theta_k L^{k}$. Since $(L^k)_{uv} = 0$ whenever $d(u,v) \gt k$, such a filter is exactly $K$-hop localized, has $K+1$ parameters, and is evaluated by $K$ sparse matrix–vector products in $O(Km)$ time — no eigendecomposition anywhere.

**Step 3 — Chebyshev basis.** Defferrard et al. use the numerically stable Chebyshev basis on the rescaled operator $\tilde{L} = \frac{2}{\lambda_{\max}}L_{\mathrm{sym}} - I \in [-1,1]$:

$$
g_\theta \star x \approx \sum_{k=0}^{K}\theta_k T_k(\tilde{L})\,x, \qquad T_{k+1}(\tilde L) = 2\tilde{L}T_k(\tilde L) - T_{k-1}(\tilde L)
$$

**Step 4 — first order, $\lambda_{\max}\approx 2$.** Kipf & Welling take $K = 1$ and approximate $\lambda_{\max} \approx 2$ (legitimate: $\lambda_{\max}(L_{\mathrm{sym}}) \le 2$ always, Topic 06). Then $\tilde{L} = L_{\mathrm{sym}} - I = -D^{-1/2}AD^{-1/2}$, and with $T_0 = I$, $T_1 = \tilde{L}$:

$$
g_\theta \star x \approx \theta_0 x + \theta_1\big({-}D^{-1/2}AD^{-1/2}\big)x
$$

**Step 5 — tie the parameters.** Setting $\theta = \theta_0 = -\theta_1$ (fewer parameters, less overfitting) gives

$$
g_\theta \star x \approx \theta\big(I + D^{-1/2}AD^{-1/2}\big)x
$$

**Step 6 — renormalization.** The operator $I + D^{-1/2}AD^{-1/2}$ has spectrum in $[0,2]$; stacking it repeatedly amplifies the top of that range and destabilizes deep models. Replace it by the self-looped normalization

$$
I + D^{-1/2}AD^{-1/2} \ \rightsquigarrow \ \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}, \qquad \tilde{A} = A + I, \ \ \tilde{D}_{vv} = \textstyle\sum_u \tilde{A}_{vu}
$$

whose spectrum lies in $(-1, 1]$. **Step 7 — vectorize over channels**: with $H^{(l)} \in \mathbb{R}^{n \times f_l}$ and weights $W^{(l)} \in \mathbb{R}^{f_l \times f_{l+1}}$,

$$
\boxed{H^{(l+1)} = \sigma\big(\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}H^{(l)}W^{(l)}\big)}
$$

$\blacksquare$

**Complexity.** One layer costs $O(m f_l + n f_l f_{l+1})$: sparse propagation plus a dense feature map, both linear in the graph size. The spectral derivation is a *motivation*; the computation is purely spatial.

### Proof 6: Over-smoothing — deep propagation collapses to rank one (T7)

**Setup.** Let $G$ be connected, $\tilde{A} = A + I$, $\tilde{D} = \mathrm{diag}(\tilde{A}\mathbf{1})$, and $\hat{S} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$. Note $\hat S = I - \tilde{L}_{\mathrm{sym}}$ where $\tilde{L}_{\mathrm{sym}}$ is the normalized Laplacian of the self-looped graph.

**Step 1 — the top eigenpair.** Put $v_1 = \tilde{D}^{1/2}\mathbf{1}/\Vert \tilde{D}^{1/2}\mathbf{1}\Vert$. Then

$$
\hat{S}\,\tilde{D}^{1/2}\mathbf{1} = \tilde{D}^{-1/2}\tilde{A}\mathbf{1} = \tilde{D}^{-1/2}\tilde{D}\mathbf{1} = \tilde{D}^{1/2}\mathbf{1}
$$

so $\mu_1 = 1$ with eigenvector $v_1$, corresponding to the eigenvalue $0$ of $\tilde{L}_{\mathrm{sym}}$.

**Step 2 — the spectrum is inside $(-1, 1]$.** By Topic 06's theorem, $\mathrm{spec}(\tilde{L}_{\mathrm{sym}}) \subseteq [0,2]$, hence $\mathrm{spec}(\hat S) \subseteq [-1,1]$. The endpoint $-1$ would require $\tilde{L}_{\mathrm{sym}}$ to have eigenvalue $2$, i.e. a bipartite component — impossible, since every vertex carries a self-loop (an odd cycle of length $1$). The endpoint $+1$ is simple because the self-looped graph is still connected, so $\dim\ker\tilde{L}_{\mathrm{sym}} = 1$. Therefore

$$
1 = \mu_1 \gt \mu_2 \ge \cdots \ge \mu_n \gt -1
$$

**Step 3 — powers converge.** Diagonalize $\hat{S} = \sum_{i}\mu_i v_iv_i^{\top}$. Then $\hat{S}^{\,l} = \sum_i \mu_i^{\,l}v_iv_i^{\top}$, and since $\vert \mu_i\vert \lt 1$ for $i \ge 2$,

$$
\big\Vert \hat{S}^{\,l} - v_1v_1^{\top}\big\Vert_2 = \max_{i \ge 2}\vert \mu_i\vert^{\,l} = \big(\max(\vert\mu_2\vert, \vert\mu_n\vert)\big)^{l} \longrightarrow 0
$$

**Step 4 — consequence for features.** With linear layers ($\sigma = \mathrm{id}$) and weights $W$, $H^{(l)} = \hat{S}^{\,l}XW^{(1)}\cdots W^{(l)}$, so

$$
\boxed{H^{(l)} \longrightarrow v_1\big(v_1^{\top}X\,\textstyle\prod_j W^{(j)}\big), \quad \text{rank } 1: \ h_v \propto \sqrt{\tilde{d}_v}}
$$

Every node's representation becomes a fixed multiple of $\sqrt{\tilde{d}_v}$ — nodes are distinguishable only by degree, and all class information is erased. $\blacksquare$

**Refinements and remedies.**

- The rate is geometric with ratio $\max(\vert\mu_2\vert, \vert\mu_n\vert)$, i.e. governed by the **spectral gap**: well-connected graphs over-smooth *faster*, which is why GCNs on expander-like graphs are shallow.
- ReLU does not save the model: Oono & Suzuki (2020) show the distance to the "collapsed" subspace still contracts geometrically when the weight norms are bounded.
- Practical remedies: residual/initial connections (GCNII), jumping-knowledge concatenation, PairNorm / normalization layers, edge dropout (DropEdge, which raises the gap), and simply keeping depth at $2$–$3$ layers.

## 4. Computational & Algorithmic Insights

### The pipeline and its costs

| Stage | Typical method | Cost | Failure mode if done badly |
|---|---|---|---|
| Build similarity graph | kNN with a KD-tree / approximate NN | $O(n\log n)$ to $O(n^2)$ | wrong $\sigma$ or $k$ destroys cluster structure before any linear algebra runs |
| Choose Laplacian | $L_{\mathrm{rw}}$ by default | — | unnormalized on heavy-tailed degrees splits off hubs |
| Extract $k$ eigenvectors | Lanczos / LOBPCG on the sparse operator | $O(\text{iter}\cdot m k)$ | eigenvector mixing when eigenvalues are nearly degenerate |
| Round | $k$-means with $k$-means++ init, several restarts | $O(nk^2\cdot\text{iter})$ | bad local optimum; always restart |
| Choose $k$ | eigengap heuristic $\arg\max_k(\lambda_{k+1}-\lambda_k)$ | free | no clear gap means no clear cluster count — report that |

**Scaling.** Beyond $n \approx 10^6$, replace the exact eigensolve by the Nyström method (sample columns, extend eigenvectors) or by power-iteration clustering; or coarsen the graph first with effective-resistance sparsification (Topic 06).

### Choosing the similarity graph — the decision that actually matters

- **Gaussian kernel bandwidth $\sigma$**: too small and the graph disconnects into singletons ($\lambda_2 = 0$, meaningless eigenvectors); too large and everything is one blob. A robust default is the *local scaling* $w_{ij} = \exp(-\Vert x_i - x_j\Vert^2/(\sigma_i\sigma_j))$ with $\sigma_i$ the distance to $i$'s $7$-th nearest neighbour (Zelnik-Manor & Perona).
- **kNN vs. $\varepsilon$-ball**: kNN adapts to varying density and is the standard choice; $\varepsilon$-balls produce disconnected graphs on non-uniform data.
- **Mutual kNN** keeps clusters of different densities separate but disconnects more easily — useful when the clusters genuinely have different scales.
- **Sparsity is a feature**: a sparse graph makes the eigenproblem fast *and* regularizes; a fully connected Gaussian graph is $O(n^2)$ and usually no better.

**Diagnostic.** Before clustering, count connected components (Topic 06). If the graph already has $c \ge k$ components, spectral clustering degenerates: the bottom eigenspace is spanned by component indicators and carries no within-component information.

### GNN implementation notes

- **Never diagonalize.** $\hat{S}H$ is a sparse matrix product. The "spectral" story is a derivation, not an algorithm.
- **Normalization variants**: $\hat{S} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ (GCN, symmetric), $\tilde{D}^{-1}\tilde{A}$ (mean aggregation, GraphSAGE), or attention-weighted $\alpha_{uv}$ (GAT). All are message passing; they differ in the aggregation weights and hence in the filter's frequency response.
- **Depth**: $2$–$3$ layers is standard; the receptive field is $l$ hops, and beyond the graph's diameter extra layers only smooth.
- **Mini-batching** on large graphs requires neighbour sampling (GraphSAGE), layer sampling (FastGCN), or subgraph sampling (Cluster-GCN — which *uses spectral clustering* to build the batches, closing the loop of this module).
- **Numerical care**: add self-loops before computing $\tilde{D}$, guard against isolated vertices ($\tilde{d}_v \ge 1$ always holds thanks to the self-loop), and store $\hat{S}$ once rather than recomputing per layer.
- **Expressivity ceiling**: message passing is bounded above by the $1$-Weisfeiler–Lehman test (Xu et al., 2019); it cannot distinguish, for example, two triangles from a hexagon under uniform features. Spectral features (Laplacian eigenvectors as positional encodings) are one standard way to break that ceiling.

### Verification strategy

- **Sanity graph**: run the pipeline on two well-separated Gaussian blobs; if it fails there, the bug is in the graph construction, not the algorithm.
- **Concentric circles**: the canonical test that separates spectral clustering from $k$-means. Spectral must succeed; $k$-means must fail.
- **Eigenvalue check**: the smallest eigenvalue must be $\approx 0$ with a near-constant eigenvector (for $L_{\mathrm{rw}}$, exactly $\mathbf{1}$). If not, the Laplacian is built wrong.
- **Objective recomputation**: after rounding, evaluate the *discrete* NCut of the produced partition and compare with the relaxed lower bound $\sum_{i\le k}\lambda_i$. A huge gap means the rounding failed, not the theory.
- **Permutation invariance**: shuffling the vertex order must not change the partition (up to relabelling) — a cheap test that catches index bugs.
- **GCN over-smoothing probe**: measure the average pairwise cosine similarity of node embeddings per layer; a rise toward $1$ with depth is the over-smoothing signature predicted by Proof 6.

## 5. Real-World Physics & AI/ML Applications

### Physics, chemistry, and the sciences

- **Metastability and Markov state models**: in molecular dynamics, conformational states are metastable sets of a transition matrix; the leading eigenvectors of $P$ (equivalently the bottom of $L_{\mathrm{rw}}$) identify them, and PCCA+ is spectral clustering under another name.
- **Image segmentation**: Shi & Malik's normalized cuts, built on pixel-similarity graphs, was the state of the art for a decade and remains the reference formulation for the segmentation objective.
- **Circuit partitioning and parallel load balancing**: spectral bisection distributes a mesh across processors while minimizing inter-processor communication — the original engineering driver for algebraic connectivity.
- **Community detection in networks**: for the stochastic block model, spectral methods on $A$, $L_{\mathrm{sym}}$, or the non-backtracking matrix recover communities down to (nearly) the information-theoretic detectability threshold.
- **Brain parcellation and connectomics**: functional connectivity graphs are partitioned spectrally into cortical regions.
- **Mesh processing**: the Laplace–Beltrami eigenfunctions of a triangle mesh give shape descriptors, segmentation, and smoothing — the continuous cousin of everything above.

### AI and machine learning

- **Node classification with GCNs**: the canonical benchmark (Cora, Citeseer, PubMed) — two GCN layers on a citation graph with bag-of-words features beat all pre-2017 methods using a fraction of the parameters.
- **Link prediction and recommendation**: graph autoencoders and PinSage-style GNNs generate embeddings whose inner products score candidate edges, deployed at web scale.
- **Molecular property prediction and drug discovery**: message-passing networks over molecular graphs (Gilmer et al.) predict quantum-chemical properties, with atoms as nodes and bonds as edges.
- **Physics simulation**: mesh-based GNNs learn to advance PDE solutions on unstructured meshes, with each layer a local message-passing stencil.
- **Traffic and spatiotemporal forecasting**: road networks with diffusion-convolutional recurrent layers — literally $g_\theta(L)$ filters applied in time.
- **Graph transformers**: Laplacian eigenvectors serve as positional encodings, letting global attention recover graph structure; sign and basis ambiguity of eigenvectors is handled by sign flipping or SignNet.
- **Semi-supervised learning without a GNN**: label propagation minimizes $x^{\top}Lx$ subject to observed labels (Topic 06) and remains a strong, hyperparameter-light baseline that GCNs sometimes fail to beat.
- **Self-supervised objectives**: contrastive graph methods implicitly learn low-frequency Laplacian structure, and several are provably equivalent to spectral embedding.

### Known failure modes — worth memorizing

| Symptom | Cause | Fix |
|---|---|---|
| One cluster is a single vertex | unnormalized objective with a very low-degree vertex | switch to NCut / $L_{\mathrm{rw}}$ |
| All eigenvectors look like indicators of tiny components | similarity graph disconnected ($\sigma$ or $\varepsilon$ too small) | increase bandwidth, use kNN with larger $k$ |
| Clusters change between runs | $k$-means local optima, or a nearly degenerate eigenvalue | restart $k$-means; inspect the eigengap |
| Good relaxed value, terrible partition | the relaxation gap is real (Guattery–Miller) | try sweep cuts, or a different objective entirely |
| GCN accuracy drops as layers are added | over-smoothing (Proof 6) | residual/initial connections, fewer layers, PairNorm |
| GNN cannot separate structurally distinct nodes | $1$-WL expressivity ceiling | add positional encodings, structural features, or higher-order models |

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| RatioCut / NCut relaxations, three algorithms | von Luxburg (2007), *Statistics and Computing* 17(4), §5 |
| Normalized cuts, image segmentation | Shi & Malik (2000), *IEEE TPAMI* 22(8) |
| Row-normalized symmetric variant | Ng, Jordan & Weiss (2002), *NeurIPS 14* |
| Ky Fan trace minimization | Fan (1949), *PNAS* 35; Bhatia, *Matrix Analysis*, Ch. III |
| Random-walk and commute-time view | Meilă & Shi (2001), *AISTATS*; Lovász (1993), "Random walks on graphs" |
| Consistency of spectral clustering | von Luxburg, Belkin & Bousquet (2008), *Annals of Statistics* 36(2) |
| Limits of the relaxation | Guattery & Miller (1998), *SIAM J. Matrix Anal. Appl.* 19(3) |
| Cheeger guarantee for the sweep cut | Chung, *Spectral Graph Theory*, Ch. 2 |
| Spectral networks, first spectral GNN | Bruna, Zaremba, Szlam & LeCun (2014), *ICLR* |
| Chebyshev localized filters | Defferrard, Bresson & Vandergheynst (2016), *NeurIPS 29* |
| GCN propagation rule and renormalization | Kipf & Welling (2017), *ICLR* |
| Message passing framework | Gilmer et al. (2017), *ICML*; Hamilton (2020), *Graph Representation Learning*, Ch. 5 |
| Over-smoothing | Li, Han & Wu (2018), *AAAI*; Oono & Suzuki (2020), *ICLR* |
| Expressivity and $1$-WL | Xu, Hu, Leskovec & Jegelka (2019), *ICLR* |
| Laplacian eigenmaps / manifold limit | Belkin & Niyogi (2003), *Neural Computation* 15(6) |

**Cross-links within this repository**

- All spectral prerequisites — $L$, $L_{\mathrm{sym}}$, $L_{\mathrm{rw}}$, Fiedler vector, Cheeger inequality: [`../06_graph_laplacian_and_spectral_theory/`](../06_graph_laplacian_and_spectral_theory/README.md)
- Matrix calculus, gradients through matrix products, and the AI-facing treatment of graph matrices and GNN layers: [`../../linear_algebra/10_matrix_calculus_graph_and_ai_applications/`](../../linear_algebra/10_matrix_calculus_graph_and_ai_applications/README.md) — the backpropagation mechanics of $H W$ and $\hat{S}H$ live there and are not repeated here.
- Eigenvalue theory, Courant–Fischer, and the spectral theorem: [`../../linear_algebra/06_eigenvalues_eigenvectors_spectral_theory/`](../../linear_algebra/06_eigenvalues_eigenvectors_spectral_theory/README.md)
- Minimum cuts, max-flow/min-cut, and why *unbalanced* cuts are easy: [`../05_flows_matchings_and_bipartite_graphs/`](../05_flows_matchings_and_bipartite_graphs/)
- Graph representations, adjacency and degree matrices: [`../01_graph_fundamentals_and_representations/`](../01_graph_fundamentals_and_representations/README.md)
- Iterative eigensolvers used in every step above: [`../../linear_algebra/09_numerical_spectrum_algorithms/`](../../linear_algebra/09_numerical_spectrum_algorithms/README.md)